# 2 · Training LoRA

The training code behind every LoRA number in the talk. It is the VPT demo's pipeline (`../vpt/train.py`: the same
three MedMNIST+ datasets, splits, 1,000-image subsets with the same seed, augmentation, schedule and evaluation),
with LoRA in place of the prompts:

| | linear probe | LoRA | full fine-tuning |
| --- | --- | --- | --- |
| trained | head | rank-8 pairs on W_q, W_v in 12 blocks + head | every weight + head |
| optimiser | SGD, momentum 0.9 | AdamW, weight decay 0.01 (as in the LoRA paper) | AdamW |
| learning rate | base LR × batch / 256, base LR picked on val | same | same |
| schedule | warm-up, then cosine decay | same | same |
| model selection | best validation balanced accuracy | same | same |

Linear probe and full fine-tuning are not retrained: they are the VPT talk's runs (`../vpt/weights`). The data
itself is shown in `../vpt/notebooks/02_data_medmnist.ipynb`.

In [ ]:
import sys, json, time, math
from pathlib import Path
ROOT = Path.cwd().parent            # demo/lora
VPT = ROOT.parent / "vpt"           # data, linear-probe and full fine-tuning runs
sys.path.insert(0, str(ROOT))
import numpy as np, torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
# ---- configuration -------------------------------------------------------------------------------
TASK   = "dermamnist"        # dermamnist | bloodmnist | pneumoniamnist
QUICK  = True                # True: 1,000 images, 12 epochs, one learning rate (about two minutes)
SUBSET = 1000
EPOCHS = 12 if QUICK else 100
SWEEP  = [0.004] if QUICK else [0.001, 0.004, 0.016]

## Data and model

`train.py` imports the data code from the VPT folder, so nothing about the data differs between the two talks.

In [ ]:
from types import SimpleNamespace
from torchvision import transforms
import train as T
sets, classes = T.build(TASK, transforms.Normalize([0.5] * 3, [0.5] * 3))
dl = T.loaders(sets, 64, 2, SUBSET, 0)
print(f"{TASK}: {len(dl['train'].dataset)} train / {len(sets['val'])} val / {len(sets['test'])} test | classes: {classes}")
args = SimpleNamespace(rank=8, alpha=8, dropout=0.1, batch=64, wd=0.01, seed=0, eval_every=3, select="bal_acc")

## The training loop

`train.run` builds a `LoRAViT`, gives AdamW only the parameters that require gradients (the pairs and the head), and
keeps the checkpoint with the best validation balanced accuracy. Printed here:

In [ ]:
import inspect
print(inspect.getsource(T.run))

## Run it

Learning-rate sweep on the validation split (skipped when `QUICK`), then training with the chosen rate, then one
score on the test split.

In [ ]:
t_start = time.time()
if len(SWEEP) > 1:
    scores = {lr: T.run(args, lr, 20, dl, "cuda", len(classes), log=False)[1]["bal_acc"] for lr in SWEEP}
    print("sweep (val balanced accuracy):", scores)
    base_lr = max(scores, key=scores.get)
else:
    base_lr = SWEEP[0]
model, val, state = T.run(args, base_lr, EPOCHS, dl, "cuda", len(classes))
model.load_state_dict({k: v for k, v in state.items() if not k.startswith("_")}, strict=False)
test = T.evaluate(model, dl["test"], "cuda", len(classes))
print(f"\nLoRA on {TASK}, {len(dl['train'].dataset)} training images, base LR {base_lr}: best val {val} | test {test}")
print(f"stored per task: {sum(v.numel() for v in state.values()) * 4 / 1e6:.2f} MB | {(time.time() - t_start) / 60:.1f} min")

## The real runs

The talk's numbers come from the full runs, launched from the command line so they survive a closed notebook
(`sh run_lora.sh`). Each writes its pairs and head to `weights/<task>_lora-r8_<1k|full>.safetensors` and its metrics
to the matching JSON. The validation curves of the DermaMNIST 1,000-image runs, LoRA beside the VPT talk's three
methods:

In [ ]:
import re, matplotlib.pyplot as plt
def curve(path):
    return [(int(m.group(1)), float(m.group(2))) for m in re.finditer(r"ep\s+(\d+).*?'bal_acc': ([\d.]+)", path.read_text(errors="ignore"))]
fig, ax = plt.subplots(figsize=(8, 4))
for name, path, color, ls in [("linear probe", VPT / "logs" / "derma_linear_1k.log", "#9EA3A8", "-"),
                              ("VPT-deep", VPT / "logs" / "derma_vpt_1k.log", "#9EA3A8", ":"),
                              ("LoRA", ROOT / "logs" / "dermamnist_lora_1k.log", "#BF3F2A", "-"),
                              ("full fine-tuning", VPT / "logs" / "derma_full_1k.log", "#1C1F23", "--")]:
    if path.exists() and curve(path):
        ax.plot(*zip(*curve(path)), label=name, color=color, ls=ls, lw=3 if name == "LoRA" else 1.5)
ax.set_xlabel("epoch"); ax.set_ylabel("val balanced accuracy %"); ax.set_title("DermaMNIST, 1,000 training images"); ax.legend()
plt.tight_layout(); plt.show()